# A Basic Agent Loop

Let's start by building a minimal agent loop, which will emphasise control flow and state. We will see that reasoning governs behaviour, while generation is only a subroutine inside that loop.

* Generation: Producing text given a prompt (stateless, one-shot)
* Reasoning: Iterative decision-making over time using intermediate outputs and state

We are treating the LLM as a conditional reasoning engine that we repeatedly query, not a one-time generator. This repeated querying and reasoning occurs over multiple iterations. So an agent loop has the following five steps:

``` Observe → Reason → Decide → Act → Update → (repeat) ```

| Stage        | Role                              |
| ------------ | --------------------------------- |
| Observe      | Read current state + environment  |
| Reason       | Interpret situation (LLM step)    |
| Decide       | Choose next action                |
| Act          | Execute action (could be trivial) |
| Update State | Persist results                   |

**Use case:** We will build a simple agent for vendor due diligence. Given a company name, the agent gathers a structured snapshot: founding profile, funding history, leadership, financials, and any risk flags, ready for a procurement or partnership review.

Due diligence is a natural multi-hop task: each answer points to the next question. The agent must decide what to look up next based on what it already knows — exactly the pattern that separates agents from single-shot LLM calls.

With this exercise, you will
- Understand each step of the agent loop as a discrete, inspectable function
- See how state accumulates across iterations as the agent builds its knowledge
- Understand how tool results feed back into reasoning
- Recognise the clean separation between the loop structure (universal) and the API call (provider-specific)

## Setup - LLM

You have three viable categories:

1. **OpenAI-style SDKs**
    * Clean, standardised interface (`responses`, `chat.completions`)
    * Strong support for structured outputs, tool calling, and streaming
    * Consistent behaviour across models

2. **Gemini SDK**
    * Slightly different abstractions (content parts, multimodal-first design)
    * Native reasoning models (e.g., Gemini 1.5/2.x)
    * More opinionated API structure

3. **Hugging Face Inference Client**
    * Model-agnostic (open weights + hosted endpoints)
    * Greater variability in output quality and formatting
    * Requires more manual control (prompt discipline, parsing)

At a systems level, all are interoperable if you abstract the interface:

```Python
def call_llm(messages: list[dict]) -> str:
    ...
```

In [94]:
# !pip install google-generativeai

import os
from datetime import date
from google import genai
from google.genai import types
# Fetch your API Key as an environment variable (or load it directly if variable naming is conventional)
from dotenv import load_dotenv
load_dotenv(override=True)
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
# embeddings and the vector DB run locally and need no key; the key is for
# generation in 2.2 and the Ragas judge in 3.1
print("GOOGLE_API_KEY loaded:", bool(GOOGLE_API_KEY))
# os.environ["GEMINI_API_KEY"] # = "..."

gem_client = genai.Client()     # store key in loaded .env as GEMINI_API_KEY
GEM_MODEL = "gemini-3.5-flash-lite"

GOOGLE_API_KEY loaded: True


In [95]:
from openai import OpenAI
gpt_client = OpenAI()
GPT_MODEL = 'gpt-4o-mini'

In [96]:
# Usually, it's better to define it after all the other steps are set up, as we will see later
def call_llm(messages, client = gem_client, llm = GEM_MODEL):
    if 'gpt' in llm:
        response = client.chat.completions.create(
            model = llm,  # or any current lightweight model
            messages=messages,
            tools = [],
            temperature=0
        )
        return response
    
    elif 'gemini' in llm:
        response = client.models.generate_content(
            model = llm,
            contents = messages,
            config=types.GenerateContentConfig(temperature = 0, tools = [])
        )
        return response

---
## The Task

We give the agent a single natural-language goal. It must plan and execute the information-gathering steps on its own.


In [97]:
TASK = """Compile a due-diligence snapshot for vendor 'Arion Data Systems'.
I need: founding year, approximate headcount, latest funding round "(amount + date + lead investor), 
CEO name, reported ARR if available, and any legal or compliance red flags. 
Include today's date in the snapshot."""

This task has some implicit requirements:
- Multi-field extraction
- Partial knowledge handling
- Structured summarisation
- Uncertainty (hallucination)

## State

The agent does not 'think', it reads and writes state. The agent's **state** is the object that persists across loop iterations.

A minimal state object holds:
- the **goal** (never changes)
- the **message history**, the agent's working memory, grows every step
- a **step counter** and **max_steps** cap to prevent infinite loops
- a **done flag** and **final_answer** slot

The LLM is invoked with state as input and produces updates to state. Everything the agent has seen and done lives in `messages`. This is what gets sent to the LLM on every call.

In [98]:
class AgentState:
    def __init__(self, goal: str, max_steps: int = 10):
        self.goal         = goal
        self.messages     = []
        self.step_count   = 0
        self.max_steps    = max_steps
        self.done         = False
        self.final_answer = None

    def __repr__(self):
        return (
            f"AgentState(step={self.step_count}/{self.max_steps}, "
            f"done={self.done}, messages={len(self.messages)})"
        )

1. `goal`: Immutable objective, passed into every reasoning step. It acts as a global constraint.
2. `messages`: The working memory which stores the trajectory of reasoning. Its format aligns with the LLM API.
3. `step_count` / `max_steps`: A hard control on loop execution to prevent infinite loops. It acts as a primitive termination condition.
4. `done`: A binary termination flag, controlled either by step limit or model signalling completion
5. `final_answer`: Extracted terminal output which separates process from result

### Initialise the Agent

In [99]:
state = AgentState(goal=TASK, max_steps=5)

# state.add_user_message(f"Task:\n{state.goal}")

---
## Tools

Tools are plain Python functions. We write a schema for each so the LLM knows what arguments to pass and when each tool is appropriate.

Let's create some functions to use as tools

**Note on simulated data:** The tools below return realistic but pre-programmed responses for "Arion Data Systems". In production, replace these with real API calls (Companies House, Crunchbase, a news API). The agent loop itself does not change.

In [100]:
# Tool implementations 

def get_company_profile(company_name: str) -> str:
    # Returns basic company profile: founding year, headcount, HQ, sector.
    if "arion" in company_name.lower():
        return (
            "Arion Data Systems | Founded: 2017 | Employees: ~340 | HQ: Amsterdam, NL | "
            "Sector: B2B SaaS -- data pipeline tooling for regulated industries | "
            "Products: ArcPipeline (ETL), ArcVault (compliance data store)"
        )
    return f"No profile found for '{company_name}'"

In [101]:
def get_funding_history(company_name: str) -> str:
    # Returns funding rounds with amounts, dates, and lead investors.
    if "arion" in company_name.lower():
        return (
            "Series B: $28M (March 2023, Balderton Capital) | "
            "Series A: $9M (Oct 2020, Seedcamp) | "
            "Seed: $1.5M (2018, angel) | Total raised: $38.5M | Status: Private"
        )
    return f"No funding data for '{company_name}'"

In [102]:
def search_news(company_name: str, topic: str) -> str:
    # Search recent news about a company on a specific topic.
    key = f"{company_name} {topic}".lower()
    if "arion" in key:
        if any(t in key for t in ("ceo", "leadership", "executive", "founder")):
            return (
                "CEO: Jonas Meier (co-founder, formerly Palantir). "
                "CTO: Priya Natarajan (joined 2021 from AWS). No recent changes."
            )
        if any(t in key for t in ("revenue", "arr", "financial", "growth")):
            return (
                "Reported ARR EUR 12M (Q4 2023 investor deck). "
                "NRR 118%. No audited public accounts (private company)."
            )
        if any(t in key for t in ("legal", "gdpr", "compliance", "risk", "red flag")):
            return (
                "Open GDPR complaint with Dutch DPA (Feb 2023) -- cross-border transfer concern. "
                "IP dispute with former engineer settled 2022. No other litigation."
            )
        if any(t in key for t in ("customer", "client", "partner")):
            return (
                "Customers: ING Bank (NL), Medicover (PL), unnamed UK insurer. "
                "Microsoft Azure partner. SOC 2 Type II certified."
            )
        return "No significant news for that topic."
    return f"No news for '{company_name}' on '{topic}'"

In [103]:
def get_current_date() -> str:
    # Returns today's date in YYYY-MM-DD format.
    return str(date.today())

In [104]:
TOOLS = {
    "get_company_profile": get_company_profile,
    "get_funding_history": get_funding_history,
    "search_news":         search_news,
    "get_current_date":    get_current_date,
}

# Quick sanity-check
print(get_company_profile("Arion Data Systems"))
print(get_funding_history("Arion Data Systems"))
print(search_news("Arion Data Systems", "CEO leadership"))
print(search_news("Arion Data Systems", "GDPR legal risk"))
print(get_current_date())

Arion Data Systems | Founded: 2017 | Employees: ~340 | HQ: Amsterdam, NL | Sector: B2B SaaS -- data pipeline tooling for regulated industries | Products: ArcPipeline (ETL), ArcVault (compliance data store)
Series B: $28M (March 2023, Balderton Capital) | Series A: $9M (Oct 2020, Seedcamp) | Seed: $1.5M (2018, angel) | Total raised: $38.5M | Status: Private
CEO: Jonas Meier (co-founder, formerly Palantir). CTO: Priya Natarajan (joined 2021 from AWS). No recent changes.
Open GDPR complaint with Dutch DPA (Feb 2023) -- cross-border transfer concern. IP dispute with former engineer settled 2022. No other litigation.
2026-09-07


### Tool Schema

We already know how to define tool schemas for Gemini, using `types.FunctionDeclaration`s

In [105]:
# Tool schemas - Gemini format 
from google.genai import types

get_company_profile_fn = types.FunctionDeclaration(
    name="get_company_profile",
    description=(
        "Returns company profile: founding year, headcount, HQ, sector, key products. "
        "Use this first to orient yourself on an unknown vendor."
    ),
    parameters={
        "type": "object",
        "properties": {
            "company_name": {
                "type": "string",
                "description": "Full company name."
            }
        },
        "required": ["company_name"]
    }
)

get_funding_history_fn = types.FunctionDeclaration(
    name="get_funding_history",
    description=(
        "Returns funding rounds with amounts, dates, and lead investors. "
        "Use to assess financial backing and growth trajectory."
    ),
    parameters={
        "type": "object",
        "properties": {
            "company_name": {
                "type": "string",
                "description": "Full company name."
            }
        },
        "required": ["company_name"]
    }
)

search_news_fn = types.FunctionDeclaration(
    name="search_news",
    description=(
        "Search recent news about a company on a specific topic. "
        "Use targeted topics: 'CEO and leadership', 'revenue and ARR', "
        "'GDPR legal risk', 'customer contracts'. One call per topic."
    ),
    parameters={
        "type": "object",
        "properties": {
            "company_name": {
                "type": "string",
                "description": "Full company name."
            },
            "topic": {
                "type": "string",
                "description": (
                    "Specific topic, e.g. 'CEO and leadership' or 'GDPR legal risk'."
                )
            }
        },
        "required": ["company_name", "topic"]
    }
)

get_current_date_fn = types.FunctionDeclaration(
    name="get_current_date",
    description="Returns today's date in YYYY-MM-DD format.",
    parameters={
        "type": "object",
        "properties": {}
    }
)

In [106]:
# combine the function declarations into tools
TOOL_DEFINITIONS = types.Tool(function_declarations = [get_company_profile_fn, get_funding_history_fn,
                                                       search_news_fn, get_current_date_fn])

# you can directly declare the functions as a list of dicts inside the function_declarations
# types.Tool(function_declarations = [{'name': "", 'description': "", 'parameters': ""}])

### LLM Call 

Now, for this demo, let's just go ahead and directly use a Gemini call instead of abstracting the LLM calls, as we are using Gemini-style tool definitions. We can define a simple function again here.

Alternatively, you can also add a tool argument to the function like: `def call_llm(messages, client = gem_client, llm = GEM_MODEL, tools = TOOL_DEFINITIONS)`.

Note that you will also need to add the tools in the `generate_content()` method (inside the LLM call function) to register them, or the LLM will end up not using these.

In [ ]:
def model(msg):
    return gem_client.models.generate_content(model='gemini-3.5-flash-lite',
                                              contents=msg,
                                              config = types.GenerateContentConfig(
                                                  system_instruction=(
                                                        "You are a corporate research assistant conducting vendor due diligence. "
                                                        "Use the available tools to gather factual information. "
                                                        "When you have enough information, produce a structured due-diligence snapshot."
                                                    ),
                                                  tools=[TOOL_DEFINITIONS])
                            )

print("Model ready with", len(TOOL_DEFINITIONS.function_declarations), "tools.")

Model ready with 4 tools.


---
## Observe & Reason

The **Observe** step initialises the message history with the agent's goal. On the first iteration this constructs the first user message. On later iterations, observations are already in history from the Update step.

The **Reason** step sends the message history to the LLM and receives a response. The LLM either requests tool calls (needs more information) or gives a final answer. This is the only step that touches the Gemini API.


In [108]:
state

AgentState(step=0/5, done=False, messages=0)

In [109]:
def observe(state):
    if not state.messages:
        state.messages.append({"role": "user", "parts": [{"text": state.goal}]})
        print(f"[OBSERVE] Injected goal.")
    else:
        print(f"[OBSERVE] State already initialised.")


observe(state)

[OBSERVE] Injected goal.


In [110]:
state

AgentState(step=0/5, done=False, messages=1)

In [111]:
def reason(state):
    print(f"\n[Step {state.step_count + 1}] Calling LLM...")
    return model(state.messages)

---
## Decide

The **Decide** step inspects the response to determine what to do next.

If the response contains `function_call` parts → the agent needs to use tools.
If not → the agent has a final text answer.


In [112]:
def decide(response):
    parts = response.candidates[0].content.parts
    tool_calls = []
    for p in parts:
        func = getattr(p, "function_call", None)
        if func and getattr(func, "name", None):
            tool_calls.append({"name": func.name, "args": dict(func.args)})
    return ("tool_calls", tool_calls) if tool_calls else ("final_answer", response.text)

---
## Act

The **Act** step executes the requested tool calls against our Python functions. This step is completely provider-agnostic — it just calls Python and collects results.


In [113]:
def act(tool_calls):
    results = []
    for tc in tool_calls:
        name, args = tc["name"], tc["args"]
        result = TOOLS[name](**args) if name in TOOLS else f"Error: unknown tool '{name}'"
        print(f"  [{name}] {args}")
        print(f"    => {result[:100]}")
        results.append({"name": name, "result": result})
    return results

---
## Update

The **Update** step folds the tool results back into the message history so the LLM sees them on the next Reason call.

Two messages are appended:
1. The model's response (containing the `function_call` parts): stored as the raw `Content` object
2. A user message with `function_response` parts (one per tool result)

In [114]:
def update(state, response, tool_results):
    state.messages.append(response.candidates[0].content)
    state.messages.append({"role": "user", "parts": [
        {"function_response": {"name": tr["name"], "response": {"result": tr["result"]}}}
        for tr in tool_results
    ]})
    state.step_count += 1

---
## The Loop

Connect all five steps into a `run_agent()` function.

```Python
observe()                     # once, at the start
while not done and not max_steps:
    response  = reason()      # call the LLM
    decision  = decide()      # parse the response
    if tool_calls:
        results = act()       # execute tools
        update()              # fold results into history
    else:
        done = True           # final answer reached
```

Only `reason()`, `decide()`, and `update()` contain provider-specific code.
The loop shell itself is identical regardless of which LLM you use.


In [115]:
def run_agent(state):
    print("=" * 65)
    print(f"GOAL: {state.goal[:100]}...")
    print("=" * 65)
    observe(state)
    while not state.done:
        if state.step_count >= state.max_steps:
            state.final_answer = "Max steps reached."
            state.done = True
            break
        response = reason(state)
        dtype, dval = decide(response)
        if dtype == "final_answer":
            state.final_answer = dval
            state.done = True
        else:
            tool_results = act(dval)
            update(state, response, tool_results)
    print("\n" + "=" * 65)
    print("RECOMMENDATION:")
    print(state.final_answer)
    print("=" * 65)
    return state.final_answer

---
## Run It

Everything is wired up. Run the agent and watch the loop execute step by step.


In [116]:
answer = run_agent(state)

GOAL: Compile a due-diligence snapshot for vendor 'Arion Data Systems'.
I need: founding year, approximate...
[OBSERVE] State already initialised.

[Step 1] Calling LLM...


ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-3s.5-flash-lite is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

---
## Inspect the State

After the loop, we can inspect every message in the history, every
observation, reasoning step, tool call, and result the agent accumulated.


In [35]:
print(f"Steps taken:         {state.step_count}")
print(f"Messages in history: {len(state.messages)}")
print()

for i, msg in enumerate(state.messages):
    if isinstance(msg, dict):
        role, parts = msg["role"].upper(), msg["parts"]
    else:
        role, parts = msg.role.upper(), msg.parts

    for part in parts:
        if isinstance(part, dict):
            if "text" in part:
                print(f"[{i}] {role}: {part['text'][:100]}")
            elif "function_response" in part:
                fr = part["function_response"]
                print(f"[{i}] {role} tool_result: {fr['name']}() => {str(fr['response'])[:80]}")
        else:
            if hasattr(part, "text") and part.text:
                print(f"[{i}] {role}: {part.text[:100]}")
            elif hasattr(part, "function_call") and part.function_call.name:
                fc = part.function_call
                print(f"[{i}] {role} tool_call: {fc.name}({dict(fc.args)})")
    print()


Steps taken:         1
Messages in history: 3

[0] USER: Compile a due-diligence snapshot for vendor 'Arion Data Systems'.
I need: founding year, approximate

[1] MODEL tool_call: get_company_profile({'company_name': 'Arion Data Systems'})
[1] MODEL tool_call: get_funding_history({'company_name': 'Arion Data Systems'})
[1] MODEL tool_call: search_news({'company_name': 'Arion Data Systems', 'topic': 'CEO and leadership'})
[1] MODEL tool_call: search_news({'company_name': 'Arion Data Systems', 'topic': 'revenue and ARR'})
[1] MODEL tool_call: search_news({'company_name': 'Arion Data Systems', 'topic': 'GDPR legal risk'})
[1] MODEL tool_call: get_current_date({})

[2] USER tool_result: get_company_profile() => {'result': 'Arion Data Systems | Founded: 2017 | Employees: ~340 | HQ: Amsterdam
[2] USER tool_result: get_funding_history() => {'result': 'Series B: $28M (March 2023, Balderton Capital) | Series A: $9M (Oct 
[2] USER tool_result: search_news() => {'result': 'CEO: Jonas Meier (co-fo

## Summary
1. **The loop is the same regardless of LLM.** Observe → Reason → Decide → Act → Update
   is a universal pattern. Only the three provider-specific functions change.

2. **State is just an accumulating list of messages.** Everything the agent knows
   is in `state.messages`. There is no hidden memory, all context is explicit and inspectable.

3. **Each tool call narrows the next decision.** The agent gathers one piece of
   information, reasons about what is still missing, then decides the next action.
   This is the reactive intelligence that makes agents useful.

4. **Tools are decoupled from the loop.** Swapping a simulated tool for a real API
   (Crunchbase, Companies House, NewsAPI) requires changing only the tool
   implementation, not the loop.

5. **The step cap is essential.** Without `max_steps`, a confused agent can loop
   indefinitely. Always set a budget.
